In [51]:
# Author: Alyssa Feger
# 4/10/26
from recipe import transform_list
import pandas as pd

class Recipe:
    def __init__(self):
        self.materials = {}
        self.recipe_list = {}

    def import_recipes(self, material_list: list[str]):
        self.materials = transform_list(material_list)


    def to_string(self, thing_dict: dict[str, int]) -> str:
        change_str = ""
        for mat in thing_dict.keys():
            material = mat[10:]
            material = material.split("_")
            material = " ".join(material)
            string = f"{thing_dict[mat]} {material}, "
            change_str = change_str + string
    
        change_str = change_str[:-2]
    
        return change_str

    def raw_materials(self):
        recipe_df = pd.read_csv("recipes.tsv", sep="\t")  # Makes a dataframe out of the recipe data
        self.recipe_list = self.raw_materials_helper_(self.materials, recipe_df)  
    
        # Makes the return dictionary into a easy to read string
    
    
    def raw_materials_helper_(self, materials_dict: dict[str, int], recipe_df):
        mat_dict: dict[str, int] = {}
        
        for material in materials_dict.keys():
            thing_dict: dict[str, int] = {}  # Dictionary for material's recipe
            materials_mask = recipe_df.loc[:, "Craftable"] == material 
            material_df = recipe_df.loc[materials_mask, :]  # Creates a dataframe specifically for needed material
            
            if material_df.empty:  # iF material_df is empty then material was not found in dataframe
                print(f"{material} does not exist in recipe list")
            else:
                index = list(material_df.index)[0]  # Finding index of material in material_df to use indexing
                material_num = materials_dict[material]
                count = int(material_df.loc[index, "Count"])
                recipe = material_df.loc[index, "Recipe"]
    
    
                # If number of material cannot be evenly divided it does integer division and adds one
                # So that needed raw materials are not under represented
                if material_num % count != 0:
                    num_count = (material_num // count) + 1
                else:
                    num_count = material_num // count
    
                
                if recipe == "0":  # Base case for raw materials
                    if material in mat_dict.keys(): 
                        mat_dict[material] = mat_dict[material] + materials_dict[material]  # If item in mat_dict then count is added to count already in
                    else:
                        mat_dict[material] = materials_dict[material]
                else:
                    recipe_list = recipe.split(",")  # Splits the recipe list to get each item
                    for thing in recipe_list:
                        thing_split = thing.split()  # Splits the item to get its number and material name
                        mat_count = int(thing_split[0]) * num_count  
                        thing_dict[thing_split[1]] = mat_count  # Adds item and it's count to thing_dict
    
                    raw_dict = self.raw_materials_helper_(thing_dict, recipe_df)  # Recursively calls function on given material's recipe items until base case is called
    
                    
                    # Goes through raw_dict and see if it exists in mat_dict already
                    for raw in raw_dict.keys():
                        if raw in mat_dict.keys():
                            mat_dict[raw] = mat_dict[raw] + raw_dict[raw]  # If item in mat_dict then count is added to count already in
                        else:
                            mat_dict[raw] = raw_dict[raw]  # If item not in mat_dict then raw material is added to mat_dict 
    
        return mat_dict
    

    def add_block(self, block: str):
        block_split = block.split()
        num = int(block_split[0])
        material = block_split[1:]
        material = "_".join(material)
        material2 = "minecraft:" + material

        if material2 in self.materials.keys():
            self.materials[material2] = num
        else:
            self.materials[material2] = num

    def remove_block(self, block: str):
        block_split = block.split()
        material = "_".join(block_split)
        material = "minecraft:" + material

        if material in self.materials.keys():
            self.materials.pop(material)
        else:
            print("Block is not in block list")

        
    def clear_materials(self):
        self.materials = {}

    def print_materials(self):
        print(self.to_string(self.materials))

    def print_recipe(self):
        print(self.to_string(self.recipe_list))

    def is_empty(self):
        return self.materials == {}


end = "N"
error = 0
print("Hi! Welcome to recipe generator!")
print("You can put in a list of minecraft items and get the amount of required materials needed to make the items you need!")
print("----------------")
r = Recipe()
while end != "Y":
    print("What would you like to do?")
    print("----------------")
    print("1: Input list of blocks")
    print("2: Receive recipe materials for given blocks")
    print("3: See list of blocks")
    print("4: Add a block to the list")
    print("5: Remove a block from the list")
    print("6: Clear the list")
    print("7: Exit")
    print("----------------")

    selection = input("")
    
    try:
        selection = int(selection)
    except ValueError:
        print("Input was not a number, please try again")
        print("----------------")

    if selection == 1:
        print("Please input blocks in this format:")
        print("## block, ## block")
        print("Refrain from using plural form")

        material_string = input()
        material_list = material_string.split(", ")

        r.import_recipes(material_list)

    elif selection == 2:
        if r.is_empty() is False:
            r.raw_materials()
            r.print_recipe()
        else:
            print("You have not input a list of blocks, please try again")
            error += 1
            
    elif selection == 3:
        if r.is_empty() is False:
            r.print_materials()
        else:
            print("You have not input a list of blocks, please try again")
            error += 1

    elif selection == 4:
        print("Please input block in this format:")
        print("## block")
        print("Refrain from using plural form and do not input more than one")

        block = input("")
        block_split = block.split(",")
        
        if len(block_split) > 1:
            print("You added too many blocks, please try again")
        else:
            r.add_block(block)

    elif selection == 5:
        print("Please input the block you wish to remove")
        block = input()
        
        if r.is_empty() is False:
            r.remove_block(block)
        else:
            print("You have not input a list of blocks, please try again")
            error += 1

    elif selection == 6:
        if r.is_empty() is False:
            r.clear_materials()
        else:
            print("Bro you haven't put anything in what's wrong with you?")
            error += 1

    elif selection == 7:
        end = "Y"

    if error >= 10:
        print("I am so done with you.")
        end = "Y"

Hi! Welcome to recipe generator!
You can put in a list of minecraft items and get the amount of required materials needed to make the items you need!
----------------
What would you like to do?
----------------
1: Input list of blocks
2: Receive recipe materials for given blocks
3: See list of blocks
4: Add a block to the list
5: Remove a block from the list
6: Clear the list
7: Exit
----------------


 1


Please input blocks in this format:
## block, ## block
Refrain from using plural form


 40 crafting table, 13 red concrete


What would you like to do?
----------------
1: Input list of blocks
2: Receive recipe materials for given blocks
3: See list of blocks
4: Add a block to the list
5: Remove a block from the list
6: Clear the list
7: Exit
----------------


...

 6


What would you like to do?
----------------
1: Input list of blocks
2: Receive recipe materials for given blocks
3: See list of blocks
4: Add a block to the list
5: Remove a block from the list
6: Clear the list
7: Exit
----------------


 3


You have not input a list of blocks, please try again
What would you like to do?
----------------
1: Input list of blocks
2: Receive recipe materials for given blocks
3: See list of blocks
4: Add a block to the list
5: Remove a block from the list
6: Clear the list
7: Exit
----------------


 7
